# n1_train_unconditional_diffusion.ipynb

This notebook trains an **unconditional diffusion model** on 50,000 RGBA pixel art sprites (128×128, 4-view format). The model learns to generate diverse character sprites **without text conditioning**, serving as the baseline for the PIXEL-T2I text-to-image project. Training takes 3-4 hours on A100 GPU and outputs checkpoints to Google Drive for later inference.

## Section 1. Setup and Imports

In [ ]:
# Install required packages
!pip install -q accelerate

# Standard library
import os
import math
import time
import warnings
from pathlib import Path
from typing import List
import csv
from datetime import datetime

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils as nn_utils
from torch.utils.data import Dataset, DataLoader, Subset
from torch.cuda.amp import autocast, GradScaler

# PyTorch optimization
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

# Vision / Image processing
from torchvision.utils import save_image, make_grid
from PIL import Image
import torchvision.transforms as T

# Visualization
import matplotlib.pyplot as plt

# Progress bar
from tqdm.auto import tqdm

print("All libraries imported successfully")

## Section 2. Environment & Device Setup

In [ ]:
# 2.1 Environment check
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2.2 Mount Google Drive and prepare dataset
from google.colab import drive
import shutil
import zipfile

drive.mount('/content/drive')

# Drive paths
DRIVE_ROOT = Path("/content/drive/MyDrive/PIXEL-T2I").resolve()
DRIVE_ZIP_PATH = DRIVE_ROOT / "pixel_character_dataset" / "processed" / "dataset_4view.zip"

# Local SSD paths (fast reading)
LOCAL_DATA = Path("/content/data_cache")
LOCAL_DATASET = LOCAL_DATA / "dataset_4view"
IMAGE_DIR = LOCAL_DATASET / "images"

# Copy dataset to local SSD if not exists
if not IMAGE_DIR.exists():
    print(f"Copying dataset from Drive...")

    LOCAL_DATA.mkdir(parents=True, exist_ok=True)
    local_zip_path = Path("/content/temp_dataset.zip")

    shutil.copy(DRIVE_ZIP_PATH, local_zip_path)

    with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_DATA)

    os.remove(local_zip_path)
    print("Dataset copied to local SSD")
else:
    print("Dataset already in local SSD")

# Verify dataset
if IMAGE_DIR.exists():
    num_images = len(list(IMAGE_DIR.glob("*.png")))
    print(f"Dataset ready: {num_images} images at {IMAGE_DIR}")
else:
    raise FileNotFoundError(f"Cannot find images at {IMAGE_DIR}")

# 2.3 Set paths
ROOT = DRIVE_ROOT
DATASET_ROOT = LOCAL_DATASET

print("\nSection 2 complete")

## Section 3. Configuration

In [ ]:
CONFIG = {
    # Data paths
    "image_dir": IMAGE_DIR,

    # Model and output paths - Google Drive
    "checkpoint_dir": ROOT / "models" / "pixel_unconditional" / "checkpoints",
    "final_model_dir": ROOT / "models" / "pixel_unconditional",
    "log_dir": ROOT / "outputs" / "logs_unconditional",
    "sample_dir": ROOT / "outputs" / "samples_unconditional",

    # Image settings
    "image_size": 128,
    "in_channels": 4,  # RGBA format

    # DataLoader settings
    "batch_size": 64,
    "num_workers": 2,
    "shuffle": True,
    "pin_memory": True,

    # Diffusion parameters
    "timesteps": 1000,              # Total diffusion steps
    "beta_schedule": "linear",      # "linear" or "cosine"
    "beta_start": 0.0001,           # Starting beta value
    "beta_end": 0.02,               # Ending beta value

    # UNet architecture
    "unet_channels": 128,              # Base channel count (increased since no text conditioning)
    "channel_mult": (1, 2, 4, 4),      # Channel multipliers per resolution
    "num_res_blocks": 2,               # Residual blocks per resolution
    "attention_resolutions": (8, 16),  # Resolutions to apply self-attention
    "dropout": 0.1,                    # Dropout probability (can increase for unconditional)
    "num_heads": 8,                    # Number of attention heads

    # Training settings
    "num_epochs": 20,
    "learning_rate": 1e-4,
    "weight_decay": 0.01,
    "lr_scheduler": "cosine",          # "cosine" or "constant"
    "warmup_steps": 50,
    "gradient_clip": 1.0,
    "use_mixed_precision": True,       # Enable FP16 training

    # Training control
    "start_training": True,      # Set to True to start fresh training
    "resume_training": False,    # Set to True to resume from checkpoint

    # Logging and checkpointing
    "log_every": 50,                   # Log every N batches
    "save_every_epochs": 5,            # Save checkpoint every N epochs
    "keep_last_n": 3,                  # Keep only last N checkpoints
    "sample_every_epochs": 5,          # Generate samples every N epochs (more frequent)
    "num_samples": 4,                  # Number of samples to generate

    # Sampling settings
    "default_ddim_steps": 10,

    # Testing and debugging
    "quick_test": False,                # Enable quick test mode (subset of data)
    "quick_test_samples": 100,          # Number of samples for quick test
}

print("Configuration loaded successfully")
print(f"\nKey settings:")
print(f"  Dataset: {CONFIG['image_dir']}")
print(f"  Checkpoints: {CONFIG['checkpoint_dir']}")
print(f"  Logs: {CONFIG['log_dir']}")
print(f"  Samples: {CONFIG['sample_dir']}")
print(f"  Batch size: {CONFIG['batch_size']}")
print(f"  Epochs: {CONFIG['num_epochs']}")
print(f"  Quick test: {CONFIG['quick_test']}")

print("\nSection 3 complete")

## Section 4. Dataset & DataLoader

In [ ]:
# 4.1 Dataset for 4-view sprites (Unconditional)
class Sprite4ViewDataset(Dataset):
    """
    Unconditional sprite character dataset with RGBA images only
    """
    def __init__(self, config):
        """
        Args:
            config: Global configuration dictionary
        """
        self.config = config
        self.image_dir = Path(config["image_dir"])
        self.image_size = config["image_size"]

        # Get all PNG files
        self.image_files = sorted([f for f in os.listdir(self.image_dir) if f.endswith('.png')])
        print(f"Loaded {len(self.image_files)} images from {self.image_dir}")

        # Define transforms
        self.transform = self._get_transforms()

    def _get_transforms(self):
        """Define image transformations for RGBA sprites"""
        return T.Compose([
            # Ensure RGBA mode
            T.Lambda(lambda img: img.convert('RGBA') if img.mode != 'RGBA' else img),
            # Resize to target size
            T.Resize((self.image_size, self.image_size)),
            # Convert to tensor [0, 1]
            T.ToTensor(),
            # Normalize to [-1, 1] for all 4 channels
            T.Normalize(mean=[0.5, 0.5, 0.5, 0.5],
                       std=[0.5, 0.5, 0.5, 0.5]),
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        """
        Returns:
            dict: {'image': tensor (4, H, W), 'filename': str}
        """
        # Get image filename
        img_name = self.image_files[idx]
        img_path = self.image_dir / img_name

        try:
            image = Image.open(img_path)
            image = self.transform(image)
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            # Return a black RGBA image as fallback
            image = torch.zeros(4, self.image_size, self.image_size)

        return {
            'image': image,
            'filename': img_name
        }


# 4.2 DataLoader Creation
def create_dataloader(config):
    """
    Create training dataloader.

    Args:
        config: Global configuration dictionary

    Returns:
        train_loader: DataLoader for training
    """
    # Create dataset
    dataset = Sprite4ViewDataset(config)

    # Quick test mode
    if config.get("quick_test", False):
        n_samples = config.get("quick_test_samples", 1000)
        dataset = Subset(dataset, range(min(n_samples, len(dataset))))
        print(f"Quick test mode: using {len(dataset)} samples")

    # Create dataloader
    train_loader = DataLoader(
        dataset,
        batch_size=config["batch_size"],
        shuffle=config["shuffle"],
        num_workers=config["num_workers"],
        pin_memory=config["pin_memory"],
        drop_last=True
    )

    print(f"DataLoader created: {len(dataset)} samples, {len(train_loader)} batches")

    return train_loader


# 4.3 Create DataLoader
train_loader = create_dataloader(CONFIG)

print("\nSection 4 complete")

## Section 5. Diffusion Process

In [ ]:
# 5.1 Noise Schedule Functions
def linear_beta_schedule(timesteps, beta_start=0.0001, beta_end=0.02):
    """Linear schedule from DDPM paper."""
    return torch.linspace(beta_start, beta_end, timesteps)


def cosine_beta_schedule(timesteps, s=0.008):
    """Cosine schedule from Improved DDPM paper."""
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0.0001, 0.9999)


def get_beta_schedule(schedule_name, timesteps, beta_start=0.0001, beta_end=0.02):
    """Get beta schedule based on name."""
    if schedule_name == "linear":
        return linear_beta_schedule(timesteps, beta_start, beta_end)
    elif schedule_name == "cosine":
        return cosine_beta_schedule(timesteps)
    else:
        raise ValueError(f"Unknown beta schedule: {schedule_name}")


# 5.2 Precompute diffusion parameters
def prepare_noise_schedule(config):
    """Precompute all noise schedule parameters."""
    timesteps = config["timesteps"]

    betas = get_beta_schedule(
        config["beta_schedule"],
        timesteps,
        config["beta_start"],
        config["beta_end"]
    )

    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)

    sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
    sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)
    sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
    posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)

    noise_schedule = {
        "betas": betas,
        "alphas": alphas,
        "alphas_cumprod": alphas_cumprod,
        "alphas_cumprod_prev": alphas_cumprod_prev,
        "sqrt_alphas_cumprod": sqrt_alphas_cumprod,
        "sqrt_one_minus_alphas_cumprod": sqrt_one_minus_alphas_cumprod,
        "sqrt_recip_alphas": sqrt_recip_alphas,
        "posterior_variance": posterior_variance,
        'timesteps': timesteps,
    }

    return noise_schedule


# 5.3 Forward Diffusion Process
def extract(a, t, x_shape):
    """Extract coefficients at timesteps and reshape for broadcasting."""
    batch_size = t.shape[0]
    out = a.gather(-1, t)
    return out.reshape(batch_size, *((1,) * (len(x_shape) - 1)))


def q_sample(x_start, t, noise, noise_schedule):
    """
    Forward diffusion: q(x_t | x_0).
    Formula: x_t = sqrt(alpha_cumprod_t) * x_0 + sqrt(1 - alpha_cumprod_t) * noise
    """
    sqrt_alphas_cumprod_t = extract(noise_schedule["sqrt_alphas_cumprod"], t, x_start.shape)
    sqrt_one_minus_alphas_cumprod_t = extract(noise_schedule["sqrt_one_minus_alphas_cumprod"], t, x_start.shape)

    return sqrt_alphas_cumprod_t * x_start + sqrt_one_minus_alphas_cumprod_t * noise


# 5.4 Create noise schedule
noise_schedule = prepare_noise_schedule(CONFIG)

# Move to device
for key in noise_schedule:
    if isinstance(noise_schedule[key], torch.Tensor):
        noise_schedule[key] = noise_schedule[key].to(device)

print(f"Noise schedule ready: {CONFIG['beta_schedule']}, {CONFIG['timesteps']} steps")
print("\nSection 5 complete")

## Section 6. UNet Model

In [ ]:
# 6.1 Building Blocks
class SinusoidalPositionEmbeddings(nn.Module):
    """Sinusoidal time embeddings for timestep t."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings


class ResidualBlock(nn.Module):
    """Residual block with time embedding (no text conditioning)."""
    def __init__(self, in_channels, out_channels, time_emb_dim, dropout=0.0):
        super().__init__()

        self.norm1 = nn.GroupNorm(32, in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)

        self.time_emb_proj = nn.Linear(time_emb_dim, out_channels)

        self.norm2 = nn.GroupNorm(32, out_channels)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)

        if in_channels != out_channels:
            self.residual_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        else:
            self.residual_conv = nn.Identity()

        self.act = nn.SiLU()

    def forward(self, x, time_emb):
        residual = self.residual_conv(x)

        h = self.norm1(x)
        h = self.act(h)
        h = self.conv1(h)

        time_emb = self.act(time_emb)
        time_emb = self.time_emb_proj(time_emb)
        h = h + time_emb[:, :, None, None]

        h = self.norm2(h)
        h = self.act(h)
        h = self.dropout(h)
        h = self.conv2(h)

        return h + residual


class AttentionBlock(nn.Module):
    """Self-attention block for spatial features."""
    def __init__(self, channels, num_heads=8):
        super().__init__()
        self.channels = channels
        self.num_heads = num_heads
        self.head_dim = channels // num_heads

        assert channels % num_heads == 0

        self.norm = nn.GroupNorm(32, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, kernel_size=1)
        self.proj = nn.Conv2d(channels, channels, kernel_size=1)

    def forward(self, x):
        B, C, H, W = x.shape
        residual = x

        x = self.norm(x)

        qkv = self.qkv(x)
        qkv = qkv.reshape(B, 3, self.num_heads, self.head_dim, H * W)
        qkv = qkv.permute(1, 0, 2, 4, 3)
        q, k, v = qkv[0], qkv[1], qkv[2]

        scale = self.head_dim ** -0.5
        attn = torch.matmul(q, k.transpose(-2, -1)) * scale
        attn = F.softmax(attn, dim=-1)

        out = torch.matmul(attn, v)
        out = out.permute(0, 1, 3, 2).reshape(B, C, H, W)

        out = self.proj(out)
        return out + residual


class Downsample(nn.Module):
    """Spatial downsampling by factor of 2."""
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, stride=2, padding=1)

    def forward(self, x):
        return self.conv(x)


class Upsample(nn.Module):
    """Spatial upsampling by factor of 2."""
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        x = F.interpolate(x, scale_factor=2, mode='nearest')
        return self.conv(x)


# 6.2 Main UNet Model
class UNet(nn.Module):
    """
    UNet for unconditional diffusion models.
    Optimized for 128x128 RGBA sprite generation.
    """
    def __init__(self, config):
        super().__init__()

        self.config = config
        in_channels = config["in_channels"]
        model_channels = config["unet_channels"]
        channel_mult = config["channel_mult"]
        num_res_blocks = config["num_res_blocks"]
        attention_resolutions = config["attention_resolutions"]
        dropout = config["dropout"]
        num_heads = config["num_heads"]

        time_emb_dim = model_channels * 4

        # Time embedding
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(model_channels),
            nn.Linear(model_channels, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim),
        )

        # Initial conv
        self.conv_in = nn.Conv2d(in_channels, model_channels, kernel_size=3, padding=1)

        # Encoder
        self.encoder_blocks = nn.ModuleList()
        current_channels = model_channels
        current_resolution = config["image_size"]
        self.encoder_out_channels = [model_channels]

        for level, mult in enumerate(channel_mult):
            out_channels = model_channels * mult

            for i in range(num_res_blocks):
                block_layers = nn.ModuleList()
                block_layers.append(ResidualBlock(current_channels, out_channels, time_emb_dim, dropout))
                current_channels = out_channels

                if current_resolution in attention_resolutions:
                    block_layers.append(AttentionBlock(current_channels, num_heads))

                self.encoder_blocks.append(block_layers)
                self.encoder_out_channels.append(current_channels)

            if level != len(channel_mult) - 1:
                self.encoder_blocks.append(nn.ModuleList([Downsample(current_channels)]))
                current_resolution //= 2
                self.encoder_out_channels.append(current_channels)

        # Bottleneck
        self.mid_block1 = ResidualBlock(current_channels, current_channels, time_emb_dim, dropout)
        self.mid_attn = AttentionBlock(current_channels, num_heads)
        self.mid_block2 = ResidualBlock(current_channels, current_channels, time_emb_dim, dropout)

        # Decoder
        self.decoder_blocks = nn.ModuleList()

        for level, mult in reversed(list(enumerate(channel_mult))):
            out_channels = model_channels * mult

            for i in range(num_res_blocks + 1):
                skip_channels = self.encoder_out_channels.pop()
                block_layers = nn.ModuleList()
                block_layers.append(ResidualBlock(
                    current_channels + skip_channels,
                    out_channels,
                    time_emb_dim,
                    dropout
                ))
                current_channels = out_channels

                if current_resolution in attention_resolutions:
                    block_layers.append(AttentionBlock(current_channels, num_heads))

                self.decoder_blocks.append(block_layers)

                if level != 0 and i == num_res_blocks:
                    self.decoder_blocks.append(nn.ModuleList([Upsample(current_channels)]))
                    current_resolution *= 2

        # Output
        self.conv_out = nn.Sequential(
            nn.GroupNorm(32, current_channels),
            nn.SiLU(),
            nn.Conv2d(current_channels, in_channels, kernel_size=3, padding=1),
        )

    def forward(self, x, t):
        """
        Args:
            x: Noisy images (B, 4, 128, 128)
            t: Timesteps (B,)
        Returns:
            Predicted noise (B, 4, 128, 128)
        """
        time_emb = self.time_mlp(t)
        h = self.conv_in(x)

        encoder_features = [h]
        for block_layers in self.encoder_blocks:
            for layer in block_layers:
                if isinstance(layer, ResidualBlock):
                    h = layer(h, time_emb)
                elif isinstance(layer, AttentionBlock):
                    h = layer(h)
                elif isinstance(layer, Downsample):
                    h = layer(h)
            encoder_features.append(h)

        h = self.mid_block1(h, time_emb)
        h = self.mid_attn(h)
        h = self.mid_block2(h, time_emb)

        for block_layers in self.decoder_blocks:
            if any(isinstance(layer, ResidualBlock) for layer in block_layers):
                skip = encoder_features.pop()
                h = torch.cat([h, skip], dim=1)

            for layer in block_layers:
                if isinstance(layer, ResidualBlock):
                    h = layer(h, time_emb)
                elif isinstance(layer, AttentionBlock):
                    h = layer(h)
                elif isinstance(layer, Upsample):
                    h = layer(h)

        return self.conv_out(h)


# 6.3 Weight Initialization
def init_weights(m):
    """Proper weight initialization for stable training."""
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.GroupNorm):
        if m.weight is not None:
            nn.init.ones_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)


# 6.4 Create Model
unet = UNet(CONFIG)
unet.apply(init_weights)
unet = unet.to(device)

total_params = sum(p.numel() for p in unet.parameters())
trainable_params = sum(p.numel() for p in unet.parameters() if p.requires_grad)

print(f"UNet created: {total_params / 1e6:.2f}M parameters")
print("\nSection 6 complete")

## Section 7. Training Setup

In [ ]:
# 7.1 Optimizer and Scheduler

# Optimizer
optimizer = torch.optim.AdamW(
    unet.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    betas=(0.9, 0.999)
)
print(f"Optimizer: AdamW (lr={CONFIG['learning_rate']})")

# Learning rate scheduler
total_steps = len(train_loader) * CONFIG["num_epochs"]
warmup_steps = min(CONFIG["warmup_steps"], total_steps // 2)

if CONFIG["lr_scheduler"] == "cosine" and total_steps > warmup_steps:
    # Warmup scheduler
    warmup_scheduler = LinearLR(
        optimizer,
        start_factor=0.01,
        end_factor=1.0,
        total_iters=warmup_steps
    )

    # Cosine scheduler
    cosine_steps = total_steps - warmup_steps
    cosine_scheduler = CosineAnnealingLR(
        optimizer,
        T_max=max(1, cosine_steps),
        eta_min=CONFIG["learning_rate"] * 0.1
    )

    # Combine warmup + cosine
    scheduler = SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, cosine_scheduler],
        milestones=[warmup_steps]
    )

    print(f"Scheduler: Warmup + CosineAnnealing (warmup={warmup_steps}, total={total_steps})")
else:
    scheduler = None
    print(f"Scheduler: None")

# Mixed precision
if CONFIG["use_mixed_precision"]:
    scaler = GradScaler()
    print(f"Mixed precision: Enabled")
else:
    scaler = None
    print(f"Mixed precision: Disabled")

# Create directories
checkpoint_dir = Path(CONFIG["checkpoint_dir"])
checkpoint_dir.mkdir(parents=True, exist_ok=True)

log_dir = Path(CONFIG["log_dir"])
log_dir.mkdir(parents=True, exist_ok=True)

sample_dir = Path(CONFIG["sample_dir"])
sample_dir.mkdir(parents=True, exist_ok=True)

print(f"Directories ready")

print("\nSection 7 complete")

## Section 8. DDIM Sampling

In [ ]:
# 8.1 DDIM Sampling Functions
@torch.no_grad()
def ddim_sample_step(unet, x, t, t_prev, noise_schedule, eta=0.0):
    """Single DDIM sampling step."""
    device = x.device
    alphas_cumprod = noise_schedule['alphas_cumprod']

    # Extract alpha values
    t_index = t[0].item()
    alpha_t = alphas_cumprod[t_index]

    if isinstance(t_prev, torch.Tensor):
        if t_prev.item() >= 0:
            alpha_prev = alphas_cumprod[t_prev.item()]
        else:
            alpha_prev = torch.tensor(1.0, device=device)
    else:
        if t_prev >= 0:
            alpha_prev = alphas_cumprod[t_prev]
        else:
            alpha_prev = torch.tensor(1.0, device=device)

    # Reshape for broadcasting
    alpha_t = alpha_t.view(1, 1, 1, 1)
    alpha_prev = alpha_prev.view(1, 1, 1, 1)

    # Predict noise
    predicted_noise = unet(x, t)

    # Predict x0
    x0_pred = (x - torch.sqrt(1 - alpha_t) * predicted_noise) / torch.sqrt(alpha_t)
    x0_pred = torch.clamp(x0_pred, -1.0, 1.0)

    # Compute variance
    sigma_t = eta * torch.sqrt((1 - alpha_prev) / (1 - alpha_t)) * torch.sqrt(1 - alpha_t / alpha_prev)

    # Compute x_{t-1}
    noise = torch.randn_like(x) if eta > 0 else 0
    x_prev = torch.sqrt(alpha_prev) * x0_pred + \
             torch.sqrt(1 - alpha_prev - sigma_t**2) * predicted_noise + \
             sigma_t * noise

    return x_prev


@torch.no_grad()
def ddim_sample(unet, shape, noise_schedule, device, ddim_steps=50, eta=0.0):
    """DDIM sampling - fast and deterministic."""
    batch_size = shape[0]
    total_timesteps = noise_schedule['timesteps']

    # Create timestep sequence
    c = total_timesteps // ddim_steps
    ddim_timesteps = torch.arange(0, total_timesteps, c, device=device)
    ddim_timesteps = torch.cat([ddim_timesteps, torch.tensor([total_timesteps - 1], device=device)])

    # Start from noise
    x = torch.randn(shape, device=device)

    # Reverse process
    for i in reversed(range(len(ddim_timesteps))):
        t = ddim_timesteps[i]
        t_prev = ddim_timesteps[i - 1] if i > 0 else torch.tensor(-1, device=device)
        t_batch = t.repeat(batch_size)

        x = ddim_sample_step(unet, x, t_batch, t_prev, noise_schedule, eta)

    return torch.clamp(x, -1.0, 1.0)

In [ ]:
# 8.2 Quick Validation Sampling
def quick_validation_sample(unet, noise_schedule, device, epoch, save_dir):
    """Generate samples for quick validation during training."""
    print(f"Validation sampling at epoch {epoch}...")

    unet.eval()

    # Generate 4 samples with 10 steps
    samples = ddim_sample(
        unet=unet,
        shape=(4, 4, 128, 128),
        noise_schedule=noise_schedule,
        device=device,
        ddim_steps=CONFIG.get("default_ddim_steps", 10),
        eta=0.0
    )

    # Save
    epoch_dir = Path(save_dir) / f"epoch_{epoch}"
    save_generated_images(samples, epoch_dir, f"epoch{epoch}")

    print(f"Samples saved: {epoch_dir}")

    unet.train()
    return samples

In [ ]:
# 8.3 Save Generated Images
def save_generated_images(images, save_dir, filename_prefix="sample"):
    """Save generated images to disk."""
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    # Convert from [-1, 1] to [0, 1]
    images = (images + 1.0) / 2.0
    images = torch.clamp(images, 0.0, 1.0)

    # Save individual images
    for i, img in enumerate(images):
        filepath = save_dir / f"{filename_prefix}_{i:04d}.png"
        save_image(img, filepath)

    # Save grid
    if len(images) > 1:
        grid_img = make_grid(images, nrow=2, padding=2, pad_value=1.0)
        grid_path = save_dir / f"{filename_prefix}_grid.png"
        save_image(grid_img, grid_path)

    print(f"  Saved {len(images)} images")

## Section 9. Training Loop

In [ ]:
# 9.1 Training Loop
def train_one_epoch(epoch, unet, train_loader, optimizer, scheduler, scaler,
                    noise_schedule, device, config):
    """Train for one epoch."""
    unet.train()

    epoch_loss = 0.0
    num_batches = 0

    # Progress bar
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['num_epochs']}")

    for batch_idx, batch in enumerate(pbar):
        start_time = time.time()

        # Get images
        images = batch['image'].to(device)
        batch_size = images.shape[0]

        # Random timesteps
        t = torch.randint(0, config["timesteps"], (batch_size,), device=device).long()

        # Add noise
        noise = torch.randn_like(images)
        noisy_images = q_sample(images, t, noise, noise_schedule)

        # Predict noise
        if config["use_mixed_precision"]:
            with autocast():
                noise_pred = unet(noisy_images, t)
                loss = F.mse_loss(noise_pred, noise)
        else:
            noise_pred = unet(noisy_images, t)
            loss = F.mse_loss(noise_pred, noise)

        # Backward
        optimizer.zero_grad()

        if config["use_mixed_precision"]:
            scaler.scale(loss).backward()

            # Gradient clipping
            if config["gradient_clip"] > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(unet.parameters(), config["gradient_clip"])

            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()

            # Gradient clipping
            if config["gradient_clip"] > 0:
                torch.nn.utils.clip_grad_norm_(unet.parameters(), config["gradient_clip"])

            optimizer.step()

        # Update learning rate
        if scheduler is not None:
            scheduler.step()

        # Track loss
        epoch_loss += loss.item()
        num_batches += 1

        # Update progress bar
        current_lr = optimizer.param_groups[0]['lr']
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'avg_loss': f'{epoch_loss/num_batches:.4f}',
            'lr': f'{current_lr:.6f}'
        })

        # Periodic logging
        if batch_idx % config["log_every"] == 0:
            batch_time = time.time() - start_time
            print(f"\n[Epoch {epoch+1}] [Batch {batch_idx}/{len(train_loader)}] "
                  f"Loss: {loss.item():.4f}, Avg: {epoch_loss/num_batches:.4f}, "
                  f"LR: {current_lr:.6f}, Time: {batch_time:.2f}s")

    return epoch_loss / num_batches

In [ ]:
# 9.2 Checkpoint Management
def save_checkpoint(epoch, unet, optimizer, scheduler, loss, checkpoint_dir,
                   loss_history=None, best_loss=None, keep_last_n=3):
    """Save training checkpoint."""
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(exist_ok=True, parents=True)

    checkpoint = {
        'epoch': epoch,
        'model_state_dict': unet.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'loss': loss,
        'loss_history': loss_history,
        'config': CONFIG,
    }

    # Save epoch checkpoint
    checkpoint_path = checkpoint_dir / f"checkpoint_epoch_{epoch+1}.pt"
    torch.save(checkpoint, checkpoint_path)

    # Update latest
    latest_path = checkpoint_dir / "checkpoint_latest.pt"
    torch.save(checkpoint, latest_path)

    # Check if best
    is_best = False
    if best_loss is None or loss < best_loss:
        best_path = checkpoint_dir / "checkpoint_best.pt"
        torch.save(checkpoint, best_path)

        model_only_path = checkpoint_dir / "model_best.pt"
        torch.save(unet.state_dict(), model_only_path)

        print(f"New best model: {loss:.4f}")
        is_best = True
        best_loss = loss

    # Cleanup old checkpoints
    all_checkpoints = sorted(checkpoint_dir.glob("checkpoint_epoch_*.pt"))
    if len(all_checkpoints) > keep_last_n:
        for old_checkpoint in all_checkpoints[:-keep_last_n]:
            old_checkpoint.unlink()

    return best_loss


def load_checkpoint(checkpoint_path, unet, optimizer=None, scheduler=None, device='cuda'):
    """
    Load checkpoint and restore training state.

    Returns:
        epoch: Loaded epoch number
        loss: Loaded loss value
        loss_history: Training history
        best_loss: Best loss value
    """
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

    # Load model
    unet.load_state_dict(checkpoint['model_state_dict'])

    # Load optimizer
    if optimizer is not None and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    # Load scheduler
    if scheduler is not None and checkpoint.get('scheduler_state_dict') is not None:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    # Extract info
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']
    loss_history = checkpoint.get('loss_history', [])
    best_loss = min(loss_history) if loss_history else loss

    print(f"Checkpoint loaded: {Path(checkpoint_path).name}")
    print(f"  Epoch: {epoch+1}")
    print(f"  Loss: {loss:.4f}")
    print(f"  Best loss: {best_loss:.4f}")
    print(f"  History: {len(loss_history)} epochs")

    return epoch, loss, loss_history, best_loss


def save_loss_history(loss_history, save_path):
    """Save loss history to CSV."""
    import csv
    from datetime import datetime

    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    with open(save_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['epoch', 'loss', 'timestamp'])

        for epoch, loss in enumerate(loss_history, start=1):
            timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            writer.writerow([epoch, f'{loss:.6f}', timestamp])

    print(f"Loss history saved: {save_path.name}")

In [ ]:
# 9.3 Main Training Function
def train(unet, train_loader, optimizer, scheduler, scaler,
          noise_schedule, device, config, start_epoch=0):
    """Main training loop."""
    print(f"\nStarting training:")
    print(f"  Epochs: {config['num_epochs']}")
    print(f"  Batches/epoch: {len(train_loader)}")
    print(f"  Starting from: epoch {start_epoch+1}\n")

    loss_history = []
    best_loss = None

    try:
        for epoch in range(start_epoch, config["num_epochs"]):
            epoch_start = time.time()

            # Train one epoch
            avg_loss = train_one_epoch(
                epoch, unet, train_loader, optimizer, scheduler, scaler,
                noise_schedule, device, config
            )

            loss_history.append(avg_loss)
            epoch_time = time.time() - epoch_start

            print(f"\nEpoch {epoch+1} summary: Loss={avg_loss:.4f}, Time={epoch_time/60:.1f}min")

            # Save checkpoint
            if (epoch + 1) % config["save_every_epochs"] == 0:
                best_loss = save_checkpoint(
                    epoch, unet, optimizer, scheduler, avg_loss,
                    checkpoint_dir, loss_history, best_loss,
                    keep_last_n=config.get("keep_last_n", 3)
                )

            # Quick validation
            if (epoch + 1) % config.get("sample_every_epochs", 5) == 0:
                quick_validation_sample(
                    unet, noise_schedule, device, epoch+1, sample_dir
                )

            # Save loss CSV
            loss_csv_path = log_dir / "loss_history.csv"
            save_loss_history(loss_history, loss_csv_path)

    except KeyboardInterrupt:
        print("\nTraining interrupted")
        best_loss = save_checkpoint(
            epoch, unet, optimizer, scheduler, avg_loss,
            checkpoint_dir, loss_history, best_loss,
            keep_last_n=config.get("keep_last_n", 3)
        )
        loss_csv_path = log_dir / "loss_history.csv"
        save_loss_history(loss_history, loss_csv_path)

    # Final checkpoint
    best_loss = save_checkpoint(
        config["num_epochs"]-1, unet, optimizer, scheduler,
        loss_history[-1] if loss_history else 0.0,
        checkpoint_dir, loss_history, best_loss,
        keep_last_n=config.get("keep_last_n", 3)
    )

    loss_csv_path = log_dir / "loss_history.csv"
    save_loss_history(loss_history, loss_csv_path)

    print(f"\nTraining complete:")
    print(f"  Final loss: {loss_history[-1]:.4f}")
    print(f"  Best loss: {best_loss:.4f}")
    print(f"  CSV: {loss_csv_path}")

    return loss_history

In [ ]:
# 9.4 Start Training
START_TRAINING = CONFIG.get("start_training", False)
RESUME_TRAINING = CONFIG.get("resume_training", False)

if START_TRAINING and not RESUME_TRAINING:
    # Fresh training
    print("Starting fresh training...")

    loss_history = train(
        unet, train_loader, optimizer, scheduler, scaler,
        noise_schedule, device, CONFIG, start_epoch=0
    )

    print("Training completed")

elif RESUME_TRAINING:
    # Resume training
    print("Resuming from checkpoint...")

    checkpoint_path = checkpoint_dir / "checkpoint_latest.pt"

    if checkpoint_path.exists():
        # Use load_checkpoint function
        start_epoch, last_loss, previous_loss_history, best_loss = load_checkpoint(
            checkpoint_path, unet, optimizer, scheduler, device
        )

        print(f"Resuming from epoch {start_epoch+1}")

        # Continue training
        new_loss_history = train(
            unet, train_loader, optimizer, scheduler, scaler,
            noise_schedule, device, CONFIG, start_epoch=start_epoch+1
        )

        # Combine histories
        combined_loss_history = previous_loss_history + new_loss_history
        combined_csv_path = log_dir / "loss_history_combined.csv"
        save_loss_history(combined_loss_history, combined_csv_path)

        print("Resume training completed")
    else:
        print(f"Checkpoint not found: {checkpoint_path}")

else:
    print("Training not started")
    print("Set CONFIG['start_training'] = True to begin")

print("\nSection 9 complete")